# 실습7 · 불용어 · 동의어 · 복합어 — 사전 테이블 lexicon

형태소 분석 · 불용어 · 동의어는 pgvector 기능이 아니다. PostgreSQL 과 형태소 분석기(textsearch_ko)에는 **한국어 불용어 · 동의어 목록이 없다.**
분석기 쪽 파일(사전 파일 · mecab 사용자 사전)을 고치는 방법도 있지만 SQL 밖의 일이라, 이 실습은 **테이블 하나(`lexicon`)** 로 관리한다.

| kind | term | target | 언제 | 하는 일 |
|---|---|---|---|---|
| `compound` | 원문 표현 | (선택) 색인할 이름 | 분석 **전** | 원문에서 찾아 무조건 한 토큰으로 — 문맥에 따라 잘리는 것을 막음 |
| `stop` | 형태소 | – | 분석 **후** | 뺀다 |
| `syn` | 형태소 | 대표어 | 분석 **후** | 대표어로 바꾼다 |

흐름 : 원문 → `lex_prep`(복합어 → 자리표시) → mecab(`to_tsvector('korean')`) → `lex_norm`(자리표시 → 복합어 · 불용어 · 동의어) = `lex_tsvector`
- 테이블과 함수는 DB 최초 기동 때 `db/init/02_schema.sql` 이 만든다. 비어 있으면 `lex_tsvector(t)` = `to_tsvector('korean', t)`
- 실습1~6 의 `doc_tsv()` · `bm25_search()` 는 분석기를 그대로 쓴다. 이 실습만 `lex_tsvector` · `lex_tsquery` 로 lexicon 을 거친다
- 평가 결과는 `results/실습7평가.csv` 에 따로 쌓는다(실습3~5 비교표에 섞이지 않게)

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | 파서가 이미 버리는 것 — 조사 · 어미는 품사로 |
| 4 | lexicon 한눈에 — 항목을 넣으면 분석 결과가 어떻게 바뀌나 |
| 5 | 불용어 후보 — 코퍼스에서 가장 흔한 형태소 |
| 6 | 불용어 — 질의에서만 빼기 (문서는 그대로) |
| 7 | 동의어 — 질의만 바꾸면 생기는 일 |
| 8 | 복합어 — 형태소로 나뉘면 생기는 일 |
| 9 | 문서에 적용 — tsv 재계산 · BM25 트리거 · 평가 · 되돌리기 |
| 10 | 속도 — 분석기만 vs lexicon 을 거칠 때 |
| 11 | 정리 — 분석기 방식과 비교 |

- 셀 4 · 6 · 7 · 9 · 10 은 항목을 **트랜잭션 안에서** 넣고 끝에 되돌린다 → lexicon 은 실습 뒤에도 비어 있다
- 셀 9 는 docs 전체의 tsv 를 바꿨다가 되돌린다. 그동안 다른 노트북의 검색은 기다린다

**Run → Run All Cells** (약 2~3분 — 셀 6 약 35초 · 셀 9 약 1분~1분 30초 · 셀 10 약 30초)

### 셀 1 · 설정
- `COMPOUNDS` · `SYNONYMS` : 셀 4 · 7 · 8 · 9 에서 lexicon 에 넣을 항목 · `STOP_N` : 셀 6 에서 질의에서 뺄 불용어 수 · `STOP_APPLY` : 셀 9 에서 문서에 적용할 불용어 수

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500                                   # 평가 질문 수 (실습3·4·5 와 같은 질문)
COMPOUNDS = ["한국경제신문", "서울대컴퓨터공학과"]   # 무조건 한 토큰
SYNONYMS = [("휴대폰", "스마트폰"), ("핸드폰", "스마트폰"), ("인공지능", "ai")]   # 형태소 → 대표어
STOP_N = [0, 10, 30, 100]                      # 셀 6 : 질의에서 뺄 불용어 수 (코퍼스 문서 빈도 상위)
STOP_APPLY = 30                                # 셀 9 : 문서에도 적용할 불용어 수

### 셀 2 · 연결과 평가 도구
- `add_entries(불용어 수, 동의어, 복합어)` : 지금 트랜잭션 안에서 lexicon 에 셀 1 의 항목을 넣는다
- `TS_RANK_SQL` : 질의를 `lex_tsquery` 로 · `BM25_SQL` : `bm25_search()` 와 같은 식이지만 질의어를 `lex_tsvector` 로 — 둘 다 lexicon 을 거친다

In [ ]:
# 셀 2 · 연결과 평가 도구
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)

conn = psycopg.connect(DB_URL, autocommit=True)
conn.prepare_threshold = None      # SET 으로 바꾼 설정이 매번 반영되게 (실행 계획 재사용 끔)
register_vector(conn)              # numpy 배열 ↔ vector 타입


def sql(query, params=None):
    """SELECT 결과를 표(DataFrame)로"""
    cur = conn.execute(query, params)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description]) if cur.description else None


# 평가용 질문 : 11개마다 하나씩 고른 EVAL_N 개. 모든 노트북이 같은 질문으로 잰다
E = f"(SELECT id, doc_id AS gold, question, embedding, emb128 FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e"
RESULTS = Path("results/검색평가.csv")


def evaluate(name, query, params=None):
    """query 는 (질문 id, 정답 지문 id, 상위 문서 id 배열) 을 질문마다 한 행씩 돌려주는 SQL.
    정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균. 결과는 RESULTS 에 누적"""
    print(f"{name} …", end=" ", flush=True)          # 방법 하나가 끝날 때마다 한 줄씩 보이게
    t = time.perf_counter()
    rows = conn.execute(query, params).fetchall()
    sec = time.perf_counter() - t
    rank = [list(ids[:10]).index(g) + 1 if g in ids[:10] else None for _, g, ids in rows]
    res = {"방법": name, "Hit@10": round(np.mean([r is not None for r in rank]), 3),
           "MRR": round(np.mean([1 / r if r else 0 for r in rank]), 3), "질문수": len(rows), "초": round(sec, 1)}
    RESULTS.parent.mkdir(exist_ok=True)
    new = pd.DataFrame([res])
    if RESULTS.exists():
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old["방법"] != name], new])
    new.to_csv(RESULTS, index=False)
    print(f"Hit@10 {res['Hit@10']:.3f} · MRR {res['MRR']:.3f} · {res['초']}s", flush=True)
    return res


n_lex = conn.execute("SELECT count(*) FROM lexicon").fetchone()[0]
print("lexicon 항목", n_lex, "개", "(비어 있어야 실습3·4 와 같은 조건)" if n_lex == 0 else "← 넣어 둔 항목이 있다. 이 실습의 수치는 빈 lexicon 기준")

conn.execute("DROP TABLE IF EXISTS freq")
conn.execute("CREATE TEMP TABLE freq AS SELECT word, ndoc FROM ts_stat('SELECT tsv FROM docs')")   # 형태소별 문서 수


def add_entries(stop_n=0, syn=False, compound=False):
    """지금 트랜잭션 안에서 lexicon 에 실습 항목을 넣는다 (트랜잭션을 되돌리면 사라진다)"""
    if stop_n:
        conn.execute("INSERT INTO lexicon (kind, term) SELECT 'stop', word FROM freq ORDER BY ndoc DESC, word LIMIT %s", (stop_n,))
    if syn:
        for w, t in SYNONYMS:
            conn.execute("INSERT INTO lexicon VALUES ('syn', %s, %s)", (w, t))
    if compound:
        for c in COMPOUNDS:
            conn.execute("INSERT INTO lexicon (kind, term) VALUES ('compound', %s)", (c,))


RESULTS = Path("results/실습7평가.csv")          # 실습3~5 의 검색평가.csv 와 따로

TS_RANK_SQL = f"""SELECT e.id, e.gold, ARRAY(SELECT d.id FROM docs d WHERE d.tsv @@ x.tq
                                             ORDER BY ts_rank(d.tsv, x.tq) DESC, d.id LIMIT 10)
                  FROM {E} CROSS JOIN LATERAL (SELECT lex_tsquery(e.question, '|') AS tq OFFSET 0) x"""
# bm25_search() 와 같은 식(k1 1.2 · b 0.75). 질의어만 lex_tsvector 로 나눈다
BM25_SQL = f"""SELECT e.id, e.gold, ARRAY(
  WITH s  AS (SELECT count(*)::float8 AS n, avg(doclen)::float8 AS avgdl FROM docs),
       qt AS (SELECT DISTINCT lexeme AS term FROM unnest(lex_tsvector(e.question))),
       w  AS (SELECT qt.term, ln(1 + (s.n - d.df + 0.5) / (d.df + 0.5)) AS idf
              FROM qt JOIN bm25_df d ON d.term = qt.term CROSS JOIN s)
  SELECT t.doc_id FROM w JOIN bm25_tf t ON t.term = w.term JOIN docs d ON d.id = t.doc_id CROSS JOIN s
  GROUP BY t.doc_id
  ORDER BY sum(w.idf * t.tf * 2.2 / (t.tf + 1.2 * (0.25 + 0.75 * d.doclen / s.avgdl))) DESC, t.doc_id
  LIMIT 10) FROM {E}"""
N = conn.execute("SELECT count(*) FROM docs").fetchone()[0]
print("지문", N, "개 · 형태소 종류", conn.execute("SELECT count(*) FROM freq").fetchone()[0], "개")

### 셀 3 · 파서가 이미 버리는 것
`ts_debug` 는 문장을 조각(token)으로 나누고 각 조각이 무엇으로 색인됐는지 보여 준다.
- textsearch_ko 의 파서는 mecab 품사로 **조사 · 어미 · 접속부사를 `blank` 로 내보낸다** → 색인되지 않는다
- 한국어의 기본 "불용어 처리"는 목록이 아니라 품사로 이미 되어 있다. 이 규칙은 textsearch_ko 코드에 고정 — 설정으로 바꿀 수 없다

In [ ]:
# 셀 3 · 파서가 이미 버리는 것
S3 = "김철수 기자가 인공지능 반도체를 만들었습니다. 그리고 AI 칩은 빠르다"
D3 = sql("""SELECT alias AS 종류, token AS 조각, array_to_string(lexemes, ',') AS 색인형태소
            FROM ts_debug('korean', %s)""", (S3,))
D3 = D3[D3["조각"].str.strip() != ""].copy()
D3["결과"] = np.where(D3["종류"] == "blank", "버림 (품사)", "색인")
display(D3.set_index("조각").T)
sql("SELECT to_tsvector('korean', %s) AS tsv", (S3,))

### 셀 4 · lexicon 한눈에
셀 1 의 복합어 · 동의어와 불용어 두 개(기자 · 것)를 넣고 문장이 어떻게 분석되는지 본다. 끝에 되돌린다.
- **① 분석 전** : 복합어가 자리표시 토큰(`qq…q`)으로 바뀐다. 영문 자음만이라 mecab 이 자르지 않고, 뒤에 조사가 붙어도 한 덩어리
- **② 분석 후** : 자리표시 → 복합어, 불용어 빠짐, 동의어 → 대표어. 형태소 위치(숫자)는 그대로
- "서울대컴퓨터공학과" 는 분석기만 쓰면 문맥에 따라 `공학` 뒤의 `과` 가 붙기도 하고 빠지기도 한다 → 복합어 항목은 어느 문맥에서나 한 토큰

In [ ]:
# 셀 4 · lexicon 한눈에
display(sql("""SELECT column_name AS 컬럼, data_type AS 타입, is_nullable AS "NULL 허용"
               FROM information_schema.columns WHERE table_name = 'lexicon' ORDER BY ordinal_position"""))
S4 = ["서울대컴퓨터공학과", "그는 서울대컴퓨터공학과를 졸업했다", "한국경제신문 김철수 기자가 휴대폰 등 핸드폰 소식을 전한 것"]
with conn.transaction():
    add_entries(syn=True, compound=True)
    conn.execute("INSERT INTO lexicon (kind, term) VALUES ('stop', '기자'), ('stop', '것')")
    display(sql("SELECT kind, term, coalesce(target, '') AS target FROM lexicon ORDER BY kind, term"))
    display(sql("""SELECT s AS 문장, to_tsvector('korean', s)::text AS "분석기 그대로",
                          lex_prep(s) AS "① 분석 전 (복합어 → 자리표시)", lex_tsvector(s)::text AS "② lex_tsvector"
                   FROM unnest(%s::text[]) s""", (S4,)))
    raise psycopg.Rollback()          # 넣은 항목을 되돌린다
print("되돌린 뒤 lexicon 항목", conn.execute("SELECT count(*) FROM lexicon").fetchone()[0], "개")

### 셀 5 · 불용어 후보 — 코퍼스에서 가장 흔한 형태소
`ts_stat` 으로 형태소마다 몇 개 지문에 나오는지 센다. 상위 형태소는 거의 모든 지문에 있어 검색어로서 구별력이 없다.
- BM25 는 IDF 로 이런 형태소를 이미 0 가까이 누른다(아래 표의 idf). ts_rank 에는 IDF 가 없다

In [ ]:
# 셀 5 · 불용어 후보
F5 = sql("""SELECT word AS 형태소, ndoc AS 문서수, round(100.0 * ndoc / %(n)s, 1) AS "문서비율%%",
                   round(ln(1 + (%(n)s - ndoc + 0.5) / (ndoc + 0.5))::numeric, 3) AS idf
            FROM freq ORDER BY ndoc DESC, word LIMIT 30""", {"n": N})
fig, ax = plt.subplots(figsize=(13, 3.4))
ax.bar(F5["형태소"], F5["문서비율%"], color="#3D484A")
ax.set_ylabel("문서 비율 %"); ax.set_title(f"문서 빈도 상위 30 형태소 (지문 {N}개 중)")
plt.show()
F5.set_index("형태소").T

### 셀 6 · 불용어 — 질의에서만 빼기
상위 N 개를 `stop` 으로 넣으면 질의(`lex_tsquery` · `BM25_SQL`)에서 곧바로 빠진다. 문서 tsv 는 그대로이므로 "질의 쪽 불용어" 와 같다. 약 35초.
- 불용어 목록은 질문이 아니라 **코퍼스 빈도**로 정했다 → 채점 문항에 맞춘 값이 아니다

In [ ]:
# 셀 6 · 불용어 — 질의에서만 빼기
rows6 = []
for n in STOP_N:
    with conn.transaction():
        add_entries(stop_n=n)
        rows6.append({**evaluate(f"07 OR ts_rank 질의 불용어 {n}", TS_RANK_SQL), "불용어 수": n, "방법": "ts_rank (OR)"})
        rows6.append({**evaluate(f"07 BM25 질의 불용어 {n}", BM25_SQL), "불용어 수": n, "방법": "BM25"})
        raise psycopg.Rollback()
R6 = pd.DataFrame(rows6)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for a, m in zip(axes, ["Hit@10", "MRR"]):
    for meth, c in [("ts_rank (OR)", "#FF7043"), ("BM25", "#3D484A")]:
        s = R6[R6["방법"] == meth]
        a.plot(s["불용어 수"], s[m], "-o", color=c, label=meth)
        for x, y in zip(s["불용어 수"], s[m]):
            a.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
    a.margins(y=0.15); a.set_xlabel("질의에서 뺀 불용어 수 (코퍼스 문서 빈도 상위)"); a.set_title(m); a.legend(loc="center right")
plt.tight_layout(); plt.show()
R6.pivot(index="방법", columns="불용어 수", values=["Hit@10", "MRR"])

### 셀 7 · 동의어 — 질의만 바꾸면 생기는 일
`syn` 을 넣으면 질의는 곧바로 대표어로 바뀐다(휴대폰 → 스마트폰). 그런데 **저장된 문서 tsv 는 그대로**라 '휴대폰' 만 쓴 지문은 오히려 못 찾는다.
→ 동의어는 문서와 질의 **양쪽**에 같이 적용해야 한다 = 문서 tsv 재계산(셀 9)

In [ ]:
# 셀 7 · 동의어 — 질의만 바꾸면
Q7 = ["휴대폰", "핸드폰", "스마트폰"]
COUNT7 = """SELECT q AS 질의, lex_tsquery(q)::text AS tsquery,
                   (SELECT count(*) FROM docs, (SELECT lex_tsquery(q) AS tq OFFSET 0) x WHERE tsv @@ x.tq) AS 지문수
            FROM unnest(%s::text[]) q"""
before7 = sql(COUNT7, (Q7,)).set_index("질의")
with conn.transaction():
    add_entries(syn=True)
    after7 = sql(COUNT7, (Q7,)).set_index("질의")
    raise psycopg.Rollback()
pd.concat({"lexicon 비어 있음": before7, "syn 추가 · 문서는 그대로": after7}, axis=1)

### 셀 8 · 복합어 — 형태소로 나뉘면 생기는 일
"한국경제신문" 은 분석기가 `한국 · 경제 · 신문` 세 형태소로 자른다. 질의도 세 형태소의 AND 가 되어, 세 단어가 **따로 흩어져 있는 지문**까지 걸린다.
- 원문에 "한국경제신문" 이 실제로 있는 지문 수와 비교한다. 복합어로 등록하면 둘이 같아진다(셀 9)

In [ ]:
# 셀 8 · 복합어 — 형태소로 나뉘면
sql("""SELECT c AS 복합어, lex_tsquery(c)::text AS "형태소 질의",
              (SELECT count(*) FROM docs, (SELECT lex_tsquery(c) AS tq OFFSET 0) x WHERE tsv @@ x.tq) AS "형태소 AND 로 걸린 지문",
              (SELECT count(*) FROM docs WHERE strpos(coalesce(title, '') || ' ' || content, c) > 0) AS "원문에 실제로 있는 지문"
       FROM unnest(%s::text[]) c""", (COMPOUNDS,))

### 셀 9 · 문서에 적용 — tsv 재계산 · BM25 트리거 · 평가 · 되돌리기
한 트랜잭션 안에서 : 셀 1 의 복합어 · 동의어와 불용어 상위 `STOP_APPLY` 개를 넣고 → 지문 전체의 tsv · doclen 을 `lex_tsvector` 로 다시 계산(제목 A · 본문 D, `doc_tsv` 와 같은 가중치)(UPDATE → 실습1 의 **BM25 트리거**가 역색인도 고침) → 셀 7 · 8 을 다시 세고 평가 → **ROLLBACK**. 약 1분~1분 30초.

In [ ]:
# 셀 9 · 문서에 적용
STATE = "SELECT (SELECT count(*) FROM bm25_tf), (SELECT count(*) FROM bm25_df), (SELECT sum(doclen) FROM docs)"
state = {"적용 전": conn.execute(STATE).fetchone()}
rows9 = []
with conn.transaction():
    add_entries(stop_n=STOP_APPLY, syn=True, compound=True)
    t = time.perf_counter()
    conn.execute("""WITH src AS MATERIALIZED (SELECT id, setweight(lex_tsvector(coalesce(title, '')), 'A') ||
                                                  setweight(lex_tsvector(content), 'D') AS tsv FROM docs)
                    UPDATE docs d SET tsv = s.tsv, doclen = tsv_len(s.tsv) FROM src s WHERE d.id = s.id""")
    print(f"지문 {N}개 tsv 재계산 + BM25 트리거 : {time.perf_counter() - t:.1f}s")
    state["lexicon 적용"] = conn.execute(STATE).fetchone()
    display(sql(COUNT7, (Q7,)).set_index("질의"))
    display(sql("""SELECT c AS 복합어, lex_tsquery(c)::text AS 질의,
                          (SELECT count(*) FROM docs, (SELECT lex_tsquery(c) AS tq OFFSET 0) x WHERE tsv @@ x.tq) AS 걸린_지문,
                          (SELECT count(*) FROM docs WHERE strpos(coalesce(title, '') || ' ' || content, c) > 0) AS 원문에_있는_지문
                   FROM unnest(%s::text[]) c""", (COMPOUNDS,)))
    rows9.append({**evaluate("07 OR ts_rank lexicon 문서·질의", TS_RANK_SQL), "방법": "ts_rank (OR)"})
    rows9.append({**evaluate("07 BM25 lexicon 문서·질의", BM25_SQL), "방법": "BM25"})
    raise psycopg.Rollback()          # 항목 · tsv · 역색인을 모두 되돌린다
state["ROLLBACK 후"] = conn.execute(STATE).fetchone()
conn.execute("VACUUM (ANALYZE) docs, bm25_tf, bm25_df")
display(pd.DataFrame(state, index=["bm25_tf 행", "bm25_df 행 (형태소 종류)", "전체 형태소 수"]).T)
R9 = pd.concat([R6[R6["불용어 수"] == 0].assign(설정="lexicon 비어 있음"),
                pd.DataFrame(rows9).assign(설정=f"복합어 · 동의어 · 불용어 {STOP_APPLY}")])
R9.pivot(index="방법", columns="설정", values=["Hit@10", "MRR"])

### 셀 10 · 속도 — 분석기만 vs lexicon 을 거칠 때
지문 5,309개 전체를 분석하는 시간. lexicon 은 분석기(C 모듈) 앞뒤에 SQL 을 더하므로 느려진다. 질의는 짧아 차이가 작다.

In [ ]:
# 셀 10 · 속도
def timed(query):
    t = time.perf_counter(); conn.execute(query); return round(time.perf_counter() - t, 1)

DOCS_RAW = "SELECT sum(length(to_tsvector('korean', coalesce(title, '')) || to_tsvector('korean', content))) FROM docs"
DOCS_LEX = "SELECT sum(length(lex_tsvector(coalesce(title, '')) || lex_tsvector(content))) FROM docs"
Q_RAW = f"SELECT count(to_tsvector('korean', e.question)) FROM {E}"
Q_LEX = f"SELECT count(lex_tsvector(e.question)) FROM {E}"
T10 = [{"대상": f"지문 {N}개", "분석기만": timed(DOCS_RAW), "lexicon 비어 있음": timed(DOCS_LEX)},
       {"대상": f"질문 {EVAL_N}개", "분석기만": timed(Q_RAW), "lexicon 비어 있음": timed(Q_LEX)}]
with conn.transaction():
    add_entries(stop_n=STOP_APPLY, syn=True, compound=True)
    T10[0]["lexicon 항목 있음"] = timed(DOCS_LEX)
    T10[1]["lexicon 항목 있음"] = timed(Q_LEX)
    raise psycopg.Rollback()
pd.DataFrame(T10).set_index("대상").rename(columns=lambda c: c + " (s)")

### 셀 11 · 정리 — 분석기 방식과 비교

| 방법 | 어디서 | 편집 | SQL 안에서 | 복합어 분할 교정 | 속도 |
|---|---|---|---|---|---|
| 파서의 품사 필터 | textsearch_ko 코드 | 코드 수정 · 재빌드 | – | – | 가장 빠름 |
| tsearch 사전 파일 (synonym · simple · thesaurus) | DB 서버 `tsearch_data` 폴더 | 파일 작성 · DB 이미지 | 아님 | 형태소 단위만 | 빠름 (C) |
| mecab 사용자 사전 | DB 서버 파일 + `mecabrc` | 파일 작성 · 컴파일 · 새 연결 | 아님 | 가능 | 빠름 (C) |
| `ts_rewrite` | 질의만 | 규칙 표 | 예 | – | 빠름 |
| **lexicon 테이블 (이 실습)** | 문서 · 질의 | **INSERT · DELETE** | **예** | **가능 (원문 치환)** | 분석 시간 증가 (셀 10) |

- 어느 방법이든 **문서 쪽을 바꾸면 tsv 를 다시 계산**해야 한다. 질의만 바꾸면 셀 7 처럼 오히려 못 찾는다
- 불용어는 IDF 가 없는 ts_rank 에 효과가 있고, BM25 에는 거의 없다(셀 6)